# apatch — Экспертный туториал

- **AST-fuzzy matching** — когда код «уплыл»
- **TrustChain** — криптографическое подписание

> Все операции реальные. Ноль моков.

In [ ]:
import os, sys, json, hashlib, shutil, tempfile

SANDBOX = os.path.join(tempfile.gettempdir(), 'apatch_expert_sandbox')
if os.path.exists(SANDBOX):
    shutil.rmtree(SANDBOX)
os.makedirs(SANDBOX)
print(f'Sandbox: {SANDBOX}')

In [ ]:
from trustchain import TrustChain, TrustChainConfig

tc_dir = os.path.join(SANDBOX, '.trustchain')
cfg = TrustChainConfig(enable_chain=True, chain_storage='file', chain_dir=tc_dir)
tc = TrustChain(cfg)
receipt = tc.sign(tool_id='expert_setup', data={'action': 'init'})
print(f'TrustChain OK, sig: {receipt.signature[:50]}...')

## AST-Fuzzy на Python-классе

ИИ «забыл» комментарий. Обычный replace сломается. AST — нет.

In [ ]:
from apatch.matcher import ASTMatcher

class_file = os.path.join(SANDBOX, 'session_manager.py')
with open(class_file, 'w') as f:
    f.write(
        'class SessionManager:\n'
        '    # Base session details\n'
        '    def __init__(self, key: str):\n'
        '        self.key = key\n'
        '        self.status = "active"\n'
        '\n'
        '    def validate(self):\n'
        '        # Check status bounds\n'
        '        if self.status == "active":\n'
        '            print("Access approved")\n'
        '            return True\n'
        '        return False\n'
    )
print('Файл на диске:')
print(open(class_file).read())

In [ ]:
old_str = (
    'class SessionManager:\n'
    '    def __init__(self, key: str):\n'
    '        self.key = key\n'
    '        self.status = "active"\n'
    '\n'
    '    def validate(self):\n'
    '        if self.status == "active":\n'
    '            print("Access approved")\n'
    '            return True\n'
    '        return False'
)
new_str = (
    'class SessionManager:\n'
    '    def __init__(self, key: str):\n'
    '        self.key = key\n'
    '        self.status = "active"\n'
    '\n'
    '    def validate(self):\n'
    '        if self.status == "active":\n'
    '            print("Access approved")\n'
    '            return True\n'
    '        return False\n'
    '\n'
    '    def revoke(self):\n'
    '        self.status = "revoked"\n'
    '        print("Session revoked")'
)

matcher = ASTMatcher(class_file)
success, result, strategy = matcher.apply_patch(old_str, new_str)
print(f'Успех: {success}, Стратегия: {strategy}')

if success:
    with open(class_file, 'w') as f:
        f.write(result)
    assert '# Base session details' in result
    assert 'def revoke' in result
    print('Комментарий сохранён, метод revoke() добавлен!')

In [ ]:
with open(class_file, 'rb') as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
receipt = tc.sign(tool_id='apatch_expert', data={
    'action': 'patch_applied', 'file': 'session_manager.py',
    'strategy': strategy, 'sha256': sha256
})
print(f'TrustChain commit OK, sha256={sha256[:32]}...')

In [ ]:
shutil.rmtree(SANDBOX)
print('Sandbox удалён.')